Dataset wav files conversion into log-mel spectrograms

In [22]:
import os
import random
import shutil
import zipfile
import requests
from tqdm import tqdm
from pathlib import Path
from sklearn.model_selection import train_test_split

def download_and_prepare_speech_commands(output_dir="data", test_size=0.2, seed=42):
    """
    Downloads Google Speech Commands dataset, extracts only 'sheila' and 'others',
    and splits into train/test sets with given proportion.
    """
    url = "http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz"
    archive_path = "speech_commands_v0.02.tar.gz"
    extracted_path = "speech_commands"

    random.seed(seed)
    os.makedirs(output_dir, exist_ok=True)

    # Step 1: Download dataset if not already present
    if not os.path.exists(archive_path):
        print("Downloading dataset...")
        response = requests.get(url, stream=True)
        total_size = int(response.headers.get('content-length', 0))
        with open(archive_path, 'wb') as f, tqdm(
            desc=archive_path,
            total=total_size,
            unit='iB',
            unit_scale=True,
            unit_divisor=1024,
        ) as bar:
            for data in response.iter_content(chunk_size=1024):
                size = f.write(data)
                bar.update(size)

    # Step 2: Extract if not already extracted
    if not os.path.exists(extracted_path):
        print("Extracting dataset...")
        shutil.unpack_archive(archive_path, extracted_path)

    # Step 3: Collect sheila and others
    sheila_dir = Path(extracted_path) / "sheila"
    all_classes = [d for d in Path(extracted_path).iterdir() if d.is_dir()]
    others_dirs = [d for d in all_classes if d.name != "sheila"]

    sheila_files = list(sheila_dir.glob("*.wav"))
    others_files = []
    for d in others_dirs:
        others_files.extend(list(d.glob("*.wav")))

    print(f"Found {len(sheila_files)} 'sheila' files and {len(others_files)} 'others' files.")

    # Step 4: Train/test split
    sheila_train, sheila_test = train_test_split(sheila_files, test_size=test_size, random_state=seed)
    others_train, others_test = train_test_split(others_files, test_size=test_size, random_state=seed)

    # Step 5: Copy files into structured folders
    for split, files_sheila, files_others in [
        ("train", sheila_train, others_train),
        ("test", sheila_test, others_test),
    ]:
        for label, files in [("sheila", files_sheila), ("others", files_others)]:
            out_dir = Path(output_dir) / split / label
            out_dir.mkdir(parents=True, exist_ok=True)
            for f in files:
                shutil.copy(f, out_dir / f.name)

    print(f"Dataset prepared at: {output_dir}")
    print(f"Train -> Sheila: {len(sheila_train)}, Others: {len(others_train)}")
    print(f"Test  -> Sheila: {len(sheila_test)}, Others: {len(others_test)}")

download_and_prepare_speech_commands(output_dir="speech_data", test_size=0.2)


speech_commands_v0.02.tar.gz:   0%|          | 9.89M/2.26G [01:18<5:04:15, 132kiB/s]  


KeyboardInterrupt: 

In [37]:
import zipfile
import io
import os
import numpy as np
import librosa
import soundfile as sf
import tempfile
from tqdm import tqdm
import warnings

# === Silence noisy warnings from librosa ===
warnings.filterwarnings("ignore", category=UserWarning, module="librosa")
warnings.filterwarnings("ignore", category=FutureWarning, module="librosa")

# === Constants (matching C implementation) ===
SAMPLE_RATE = 16000
FRAME_DUR = 0.032
FRAME_SIZE = int(SAMPLE_RATE * FRAME_DUR)  # 512
FRAME_STRIDE_DUR = 0.024
FRAME_STRIDE = int(SAMPLE_RATE * FRAME_STRIDE_DUR)  # 384
NUM_BINS = FRAME_SIZE // 2  # 256
FILTER_NUMBER = 40
MIN_FREQ = 0
MAX_FREQ = SAMPLE_RATE // 2  # 8000
COEFFICIENT = 0.96875
NOISE_FLOOR = -40.0
SEGMENT_SEC = 1.0

# === C-style implementation functions ===
def pre_emphasis(audio):
    """Pre-emphasis matching C implementation"""
    emphasized = np.zeros_like(audio, dtype=np.float32)
    emphasized[0] = audio[0] / 32768.0
    for i in range(1, len(audio)):
        emphasized[i] = (audio[i] / 32768.0) - COEFFICIENT * (audio[i-1] / 32768.0)
    return emphasized

def apply_windowing(frame):
    """Hamming window matching C implementation"""
    window = 0.54 - 0.46 * np.cos(2 * np.pi * np.arange(len(frame)) / (len(frame) - 1))
    return frame * window

def hz_to_mel(hz):
    """Hz to Mel conversion matching C implementation"""
    return 1127.0 * np.log10(1 + hz / 700.0)

def mel_to_hz(mel):
    """Mel to Hz conversion matching C implementation"""
    return 700 * (10 ** (mel / 1127.0) - 1)

def create_mel_filterbank():
    """Create mel filterbank matching C implementation"""
    min_mel = hz_to_mel(MIN_FREQ)
    max_mel = hz_to_mel(MAX_FREQ)
    
    mel_points = np.zeros(FILTER_NUMBER + 2)
    mel_spacing = (max_mel - min_mel) / (FILTER_NUMBER + 1)
    for i in range(FILTER_NUMBER + 2):
        mel_points[i] = mel_to_hz(min_mel + i * mel_spacing)
        if mel_points[i] > MAX_FREQ:
            mel_points[i] = MAX_FREQ

    bin_indices = np.zeros(FILTER_NUMBER + 2, dtype=int)
    for i in range(FILTER_NUMBER + 2):
        bin_indices[i] = int(mel_points[i] * (NUM_BINS - 1) / (SAMPLE_RATE / 2.0))
        bin_indices[i] = max(0, min(NUM_BINS - 1, bin_indices[i]))

    filterbank = np.zeros((FILTER_NUMBER, NUM_BINS))

    for i in range(FILTER_NUMBER):
        left = bin_indices[i]
        middle = bin_indices[i+1]
        right = bin_indices[i+2]

        if left == middle:
            middle = min(left + 1, NUM_BINS - 1)
        if middle == right:
            right = min(middle + 1, NUM_BINS - 1)

        # Rising slope
        for j in range(left, middle):
            filterbank[i, j] = (j - left) / (middle - left)

        # Falling slope
        for j in range(middle, right):
            filterbank[i, j] = 1.0 - (j - middle) / (right - middle)
    
    return filterbank

def compute_spectrogram_c_style(audio_int16, target_frames=None):
    """
    Compute spectrogram using C-style implementation
    audio_int16: int16 audio samples
    target_frames: if specified, pad/truncate to this many frames
    """
    num_samples = len(audio_int16)
    
    # Calculate number of frames for 1 second (or available audio)
    total_duration = num_samples / SAMPLE_RATE
    num_frames_available = int((total_duration - FRAME_DUR) / FRAME_STRIDE_DUR) + 1
    
    # Use target_frames if specified, otherwise use available frames (max 42 for 1 second)
    if target_frames:
        num_frames = min(num_frames_available, target_frames)
    else:
        num_frames = min(num_frames_available, 42)  # ~1 second max
    
    # Pre-emphasis
    pre_emphasis_array = pre_emphasis(audio_int16)
    
    # Initialize spectrogram
    spectrogram = np.zeros((num_frames, NUM_BINS))

    # Frame processing
    for frame in range(num_frames):
        start = frame * FRAME_STRIDE
        end = start + FRAME_SIZE
        segment = pre_emphasis_array[start:end]
        
        # Pad if necessary
        if len(segment) < FRAME_SIZE:
            segment = np.pad(segment, (0, FRAME_SIZE - len(segment)))

        # Apply windowing
        windowed = apply_windowing(segment)
        
        # FFT
        fft = np.fft.rfft(windowed, n=FRAME_SIZE)
        magnitude = np.abs(fft)
        spectrogram[frame] = magnitude[:NUM_BINS]

    # Apply mel filterbank
    mel_filterbank = create_mel_filterbank()
    mel_spectrogram = np.dot(spectrogram, mel_filterbank.T)
    
    # Convert to log scale
    log_mel_spectrogram = 10 * np.log10(mel_spectrogram + 1e-20)

    # Apply noise floor and normalization (matching C implementation)
    log_mel_spectrogram = (log_mel_spectrogram - NOISE_FLOOR) / (-NOISE_FLOOR + 12)
    log_mel_spectrogram = np.clip(log_mel_spectrogram, 0, 1)
    
    # Quantization
    quantized = np.round(log_mel_spectrogram * 256) / 256.0
    
    # Hard threshold
    quantized = np.where(quantized >= 0.65, quantized, 0)
    
    # Pad or truncate to target_frames if specified
    if target_frames:
        if quantized.shape[0] < target_frames:
            quantized = np.pad(quantized, ((0, target_frames - quantized.shape[0]), (0, 0)))
        else:
            quantized = quantized[:target_frames]

    return quantized

# === Robust loader ===
def load_audio_from_zip(z, fname, target_sr=16000):
    try:
        with z.open(fname) as file_data:
            wav_bytes = io.BytesIO(file_data.read())

            try:
                # First try with soundfile
                y, sr = sf.read(wav_bytes, dtype="float32")
            except Exception:
                # Fallback: audioread (via temp file)
                with tempfile.NamedTemporaryFile(suffix=".wav") as tmp:
                    tmp.write(wav_bytes.getbuffer())
                    tmp.flush()
                    y, sr = librosa.load(tmp.name, sr=None)

            # Resample if needed
            if sr != target_sr:
                y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)
                sr = target_sr

            return np.asarray(y, dtype=np.float32), sr

    except Exception as e:
        print(f"[WARN] Skipping {fname} due to error: {e}")
        return None, None

# === MFE Extraction using C-style implementation ===
def extract_mfe_segments_c_style(y, sr, segment_sec=SEGMENT_SEC, target_frames=None):
    """Extract MFE segments using C-style implementation"""
    max_val = np.max(np.abs(y))
    if max_val == 0 or not np.isfinite(max_val):
        return []
    
    # Normalize to [-1, 1] range
    y = y / max_val
    if not np.all(np.isfinite(y)):
        return []
    
    # Convert to int16 format (as expected by C-style implementation)
    y_int16 = (y * 32767).astype(np.int16)
    
    mfe_segments = []
    segment_samples = int(segment_sec * sr)
    
    for start_idx in range(0, len(y_int16), segment_samples):
        end_idx = start_idx + segment_samples
        if end_idx > len(y_int16):
            # For the last segment, take what's available
            segment = y_int16[start_idx:]
            if len(segment) < segment_samples // 2:  # Skip very short segments
                break
        else:
            segment = y_int16[start_idx:end_idx]

        # Compute spectrogram using C-style implementation
        mfe = compute_spectrogram_c_style(segment, target_frames=target_frames)
        mfe_segments.append(mfe)

    return mfe_segments

# === Main processing from ZIP ===
def process_zip_dataset(zip_path, output_path, segment_sec=1.0, target_frames=None):
    samples = []
    labels = []
    label_map = {}
    skipped = 0

    with zipfile.ZipFile(zip_path, "r") as z:
        wav_files = [
            f for f in z.namelist() 
            if f.lower().endswith(".wav") and "__macosx" not in f.lower()
        ]

        for f in tqdm(wav_files, desc="Processing WAVs"):
            # Label = parent folder name
            parts = f.split("/")
            if len(parts) >= 2:
                label = parts[-2]
            else:
                label = "unknown"

            if label not in label_map:
                label_map[label] = len(label_map)

            # Load audio
            y, sr = load_audio_from_zip(z, f, target_sr=SAMPLE_RATE)
            if y is None:
                skipped += 1
                continue

            # Extract features using C-style implementation
            mfe_segments = extract_mfe_segments_c_style(y, sr, segment_sec=segment_sec, target_frames=target_frames)

            for mfe in mfe_segments:
                samples.append(mfe)
                labels.append(label_map[label])

    # Convert to numpy arrays
    samples = np.array(samples, dtype=np.float32)
    labels = np.array(labels, dtype=np.int32)

    # Save
    np.savez_compressed(
        output_path,
        features=samples,
        labels=labels,
        label_map=label_map
    )

    print("\n" + "="*50)
    print("PROCESSING DETAILED SUMMARY")
    print("="*50)
    print(f"Total WAV files found    : {len(wav_files)}")
    print(f"Total files processed    : {len(wav_files) - skipped}")
    print(f"Total files skipped      : {skipped}")
    print(f"Total valid segments     : {len(samples)}")
    print(f"Classes found            : {len(label_map)}")
    print(f"Label map                : {label_map}")
    print(f"Feature shape            : {samples.shape if len(samples) > 0 else 'N/A'}")
    print(f"\nSaved dataset: {output_path}")
    print("="*50)

# === Run ===
if __name__ == "__main__":
    process_zip_dataset(
        zip_path="datasetSheila.zip",
        output_path="sheila-mfe-1sec.npz",
        segment_sec=1.0,
        target_frames=40  # Optional: force to 40 frames per segment
    )

Processing WAVs: 100%|██████████| 4548/4548 [26:05<00:00,  2.91it/s]  



PROCESSING DETAILED SUMMARY
Total WAV files found    : 4548
Total files processed    : 4548
Total files skipped      : 0
Total valid segments     : 18578
Classes found            : 2
Label map                : {'others': 0, 'sheila': 1}
Feature shape            : (18578, 40, 40)

Saved dataset: sheila-mfe-1sec.npz


Dataset extraction for training

In [2]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

# 1. Set random seed
seed = 22
tf.random.set_seed(seed)
np.random.seed(seed)

def debug_data_distribution(X_train, y_train, X_val, y_val, X_test, y_test):
    """Debug function to check data distribution and potential issues."""
    print("\n" + "="*60)
    print("DATA DISTRIBUTION ANALYSIS")
    print("="*60)

    # Class distribution
    print("Class distribution:")
    print(f"Train: {dict(zip(*np.unique(y_train, return_counts=True)))}")
    print(f"Val:   {dict(zip(*np.unique(y_val, return_counts=True)))}")
    print(f"Test:  {dict(zip(*np.unique(y_test, return_counts=True)))}")

    # Value ranges
    print(f"\nData value ranges:")
    print(f"Train: min={X_train.min():.4f}, max={X_train.max():.4f}, mean={X_train.mean():.4f}")
    print(f"Val:   min={X_val.min():.4f}, max={X_val.max():.4f}, mean={X_val.mean():.4f}")
    print(f"Test:  min={X_test.min():.4f}, max={X_test.max():.4f}, mean={X_test.mean():.4f}")

    # Shapes
    print(f"\nData shapes:")
    print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

    # Mean activations
    train_mean = X_train.mean(axis=(1,2,3))
    val_mean = X_val.mean(axis=(1,2,3))
    print(f"\nMean activation per sample:")
    print(f"Train mean: {train_mean.mean():.4f} ± {train_mean.std():.4f}")
    print(f"Val mean:   {val_mean.mean():.4f} ± {val_mean.std():.4f}")

    # Warnings
    if len(np.unique(y_val)) != len(np.unique(y_train)):
        print("⚠️ WARNING: Validation set doesn't contain all classes!")

    if X_val.std() == 0 or X_train.std() == 0:
        print("⚠️ WARNING: Zero std detected - check your data!")

    if abs(X_train.mean() - X_val.mean()) > 0.5:
        print("⚠️ WARNING: Large mean difference between train/val sets!")

    print("="*60)

def create_balanced_dataset(shape_input, file="sheila-mfe-1sec.npz",
                           values_name='features', labels_name='labels'):
    """
    Load dataset and balance classes.
    balance_strategy = 'undersample' or 'oversample'
    """
    if len(shape_input) == 3:
        height, width, channels = shape_input
    elif len(shape_input) == 2:
        height, width = shape_input
        channels = 1
    else:
        raise ValueError("Shape Input must be (h, w) or (h, w, c).")

    # Load
    dataset = np.load(file)
    samples = dataset[values_name][:, :height, :width]
    classes = dataset[labels_name].astype(int)
    unique_classes = np.unique(classes)

    print(f"Original dataset: {samples.shape}, classes: {unique_classes}")
    print(f"Original class distribution: {dict(zip(*np.unique(classes, return_counts=True)))}")
    # Split
    X_temp, X_test, y_temp, y_test = train_test_split(
        samples, classes, test_size=0.2, stratify=classes, random_state=seed
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=seed
    )

    if channels == 1:
        X_train = np.expand_dims(X_train, -1)
        X_val = np.expand_dims(X_val, -1)
        X_test = np.expand_dims(X_test, -1)

    # Debug
    debug_data_distribution(X_train, y_train, X_val, y_val, X_test, y_test)

    return X_train, y_train, X_val, y_val, X_test, y_test, len(unique_classes)

# === Example usage ===
shape_input = (40, 40, 1)
X_train, y_train, X_val, y_val, X_test, y_test, n_classes = create_balanced_dataset(
    shape_input
)
print("Unique values in y_train:", np.unique(y_train))
print("Unique values in y_val:", np.unique(y_val))
print("Unique values in y_test:", np.unique(y_test))
X_train=X_train.reshape(X_train.shape[0], -1)
X_val=X_val.reshape(X_val.shape[0], -1)
X_test=X_test.reshape(X_test.shape[0], -1)
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Original dataset: (18578, 40, 40), classes: [0 1]
Original class distribution: {0: 16264, 1: 2314}

DATA DISTRIBUTION ANALYSIS
Class distribution:
Train: {0: 9758, 1: 1388}
Val:   {0: 3253, 1: 463}
Test:  {0: 3253, 1: 463}

Data value ranges:
Train: min=0.0000, max=1.0000, mean=0.5030
Val:   min=0.0000, max=1.0000, mean=0.5001
Test:  min=0.0000, max=1.0000, mean=0.5040

Data shapes:
Train: (11146, 40, 40, 1), Val: (3716, 40, 40, 1), Test: (3716, 40, 40, 1)

Mean activation per sample:
Train mean: 0.5030 ± 0.2454
Val mean:   0.5001 ± 0.2432
Unique values in y_train: [0 1]
Unique values in y_val: [0 1]
Unique values in y_test: [0 1]
Train: (11146, 1600), Val: (3716, 1600), Test: (3716, 1600)


In [4]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_recall_curve, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
seed = 42
tf.random.set_seed(seed)
np.random.seed(seed)

class F1Score(tf.keras.metrics.Metric):
    """Custom F1-score metric for binary classification"""
    def __init__(self, threshold=0.5, name='f1_score', **kwargs):
        super(F1Score, self).__init__(name=name, **kwargs)
        self.threshold = threshold
        self.true_positives = self.add_weight(name='tp', initializer='zeros')
        self.false_positives = self.add_weight(name='fp', initializer='zeros')
        self.false_negatives = self.add_weight(name='fn', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        # Convert probabilities to predictions
        y_pred = tf.cast(y_pred > self.threshold, tf.float32)
        y_true = tf.cast(y_true, tf.float32)
        
        # For multiclass, take argmax
        if len(y_pred.shape) > 1 and y_pred.shape[1] > 1:
            y_pred = tf.cast(tf.argmax(y_pred, axis=1), tf.float32)
            if len(y_true.shape) > 1:
                y_true = tf.cast(tf.argmax(y_true, axis=1), tf.float32)
        
        # Calculate TP, FP, FN for positive class (class 1)
        tp = tf.reduce_sum(tf.cast((y_true == 1) & (y_pred == 1), tf.float32))
        fp = tf.reduce_sum(tf.cast((y_true == 0) & (y_pred == 1), tf.float32))
        fn = tf.reduce_sum(tf.cast((y_true == 1) & (y_pred == 0), tf.float32))
        
        self.true_positives.assign_add(tp)
        self.false_positives.assign_add(fp)
        self.false_negatives.assign_add(fn)

    def result(self):
        precision = self.true_positives / (self.true_positives + self.false_positives + tf.keras.backend.epsilon())
        recall = self.true_positives / (self.true_positives + self.false_negatives + tf.keras.backend.epsilon())
        f1 = 2 * precision * recall / (precision + recall + tf.keras.backend.epsilon())
        return f1

    def reset_state(self):
        self.true_positives.assign(0)
        self.false_positives.assign(0)
        self.false_negatives.assign(0)

def create_sparse_friendly_cnn(input_shape, num_classes):
    """CNN for keyword spotting with BatchNorm"""
    inputs = tf.keras.Input(shape=input_shape, name="input")
    
    # Conv block 1
    x = layers.Conv2D(32, (3,3), padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.2)(x)

    # Conv block 2
    x = layers.Conv2D(64, (3,3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.3)(x)

    # Conv block 3
    x = layers.Conv2D(128, (3,3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.4)(x)

    # Flatten + Dense
    x = layers.Flatten()(x)
    x = layers.Dense(256, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(64, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="teacher_kws_cnn")
    
    # Use F1-score as primary metric
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy', F1Score()]
    )
    return model

def build_student_model(input_shape, num_classes=2):
    """Your exact target architecture - simple 3-layer dense network"""
    inputs = tf.keras.Input(shape=input_shape, name="input")  # Already flattened input (1600,)

    x = tf.keras.layers.Dense(256, activation="relu", name="dense_1")(inputs)
    x = tf.keras.layers.Dropout(0.3)(x)

    x = tf.keras.layers.Dense(256, activation="relu", name="dense_2")(x)
    x = tf.keras.layers.Dropout(0.3)(x)

    x = tf.keras.layers.Dense(256, activation="relu", name="dense_3")(x)
    x = tf.keras.layers.Dropout(0.3)(x)

    outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="output")(x)
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="kws_dense_model")
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy', F1Score()]
    )
    return model

def calculate_precision_recall_f1(y_true, y_pred, class_names):
    """Calculate precision, recall, and F1 for each class"""
    from sklearn.metrics import precision_recall_fscore_support
    
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )
    
    results = {}
    for i, class_name in enumerate(class_names):
        results[class_name] = {
            'precision': precision[i],
            'recall': recall[i],
            'f1': f1[i],
            'support': support[i]
        }
    
    return results

def plot_training_performance(history, model, X_test, y_test, class_names, y_train=None, y_val=None, y_test_labels=None):
    """Create comprehensive graphs showing training performance with F1-score, accuracy, and sample counts"""

    # Get predictions
    y_pred_proba = model.predict(X_test)
    y_pred = np.argmax(y_pred_proba, axis=1)
    
    # Convert one-hot encoded y_test to class indices if needed
    if len(y_test.shape) > 1:
        y_true = np.argmax(y_test, axis=1)
    else:
        y_true = y_test
    
    # Calculate precision, recall, F1 for each class
    metrics = calculate_precision_recall_f1(y_true, y_pred, class_names)
    
    # Create figure with more subplots (3x3 grid now)
    fig, axes = plt.subplots(3, 3, figsize=(22, 16))
    fig.suptitle('Model Training Performance Analysis (F1-Score Focus)', fontsize=18, fontweight='bold')
    
    # 1. F1-Score plot
    if 'f1_score' in history.history:
        axes[0, 0].plot(history.history['f1_score'], label='Training F1-Score', linewidth=2, color='green')
        axes[0, 0].plot(history.history['val_f1_score'], label='Validation F1-Score', linewidth=2, color='orange')
        axes[0, 0].set_title('F1-Score (Primary Metric)')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('F1-Score')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        axes[0, 0].set_ylim(0, 1)

    # 2. Loss plot
    axes[0, 1].plot(history.history['loss'], label='Training Loss', linewidth=2)
    axes[0, 1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
    axes[0, 1].set_title('Model Loss')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # 3. Accuracy plot
    axes[0, 2].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2, color='blue')
    axes[0, 2].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2, color='red')
    axes[0, 2].set_title('Accuracy Behavior')
    axes[0, 2].set_xlabel('Epoch')
    axes[0, 2].set_ylabel('Accuracy')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)
    
    # 4. Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    sns.heatmap(cm_percent, annot=True, fmt='.1f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=axes[1, 0])
    axes[1, 0].set_title('Confusion Matrix (% of True Class)')
    axes[1, 0].set_xlabel('Predicted Labels')
    axes[1, 0].set_ylabel('True Labels')
    
    # 5. F1-Score by class
    class_f1_scores = [metrics[c]['f1'] for c in class_names]
    bars = axes[1, 1].bar(range(len(class_names)), class_f1_scores, alpha=0.7)
    axes[1, 1].set_title('F1-Score by Class')
    axes[1, 1].set_ylim(0, 1)
    axes[1, 1].set_xticks(range(len(class_names)))
    axes[1, 1].set_xticklabels(class_names)
    for bar, value in zip(bars, class_f1_scores):
        axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                       f'{value:.3f}', ha='center', va='bottom')
    
    # 6. Precision-Recall-F1 grouped bars
    precision_values = [metrics[c]['precision'] for c in class_names]
    recall_values = [metrics[c]['recall'] for c in class_names]
    f1_values = [metrics[c]['f1'] for c in class_names]
    width = 0.25
    idx = np.arange(len(class_names))
    axes[1, 2].bar(idx - width, precision_values, width, label="Precision")
    axes[1, 2].bar(idx, recall_values, width, label="Recall")
    axes[1, 2].bar(idx + width, f1_values, width, label="F1")
    axes[1, 2].set_title('Class-wise Metrics')
    axes[1, 2].set_xticks(idx)
    axes[1, 2].set_xticklabels(class_names)
    axes[1, 2].legend()
    axes[1, 2].set_ylim(0, 1)
    
    # 7. Sample counts (stacked)
    unique, counts = np.unique(y_true, return_counts=True)
    axes[2, 0].bar(unique, counts, color=['blue', 'orange'], alpha=0.7)
    axes[2, 0].set_title('Test Set Class Distribution')
    axes[2, 0].set_xlabel('Class')
    axes[2, 0].set_ylabel('Count')
    axes[2, 0].set_xticks(unique)
    axes[2, 0].set_xticklabels(class_names)
    
    # 8. Text metrics summary
    metrics_text = ""
    if 'val_f1_score' in history.history: 
        metrics_text += f"Final Validation F1-Score: {history.history['val_f1_score'][-1]:.4f}\n" 
        metrics_text += f"Final Training F1-Score: {history.history['f1_score'][-1]:.4f}\n" 
        metrics_text += f"Best Validation F1-Score: {max(history.history['val_f1_score']):.4f}\n\n" 
    metrics_text += f"Final Validation Accuracy: {history.history['val_accuracy'][-1]:.4f}\n" 
    metrics_text += f"Final Training Accuracy: {history.history['accuracy'][-1]:.4f}\n\n" 
    
    for c in class_names: 
        metrics_text += f"{c.upper()}:\n" 
        metrics_text += f" Precision: {metrics[c]['precision']:.4f}\n" 
        metrics_text += f" Recall: {metrics[c]['recall']:.4f}\n" 
        metrics_text += f" F1-score: {metrics[c]['f1']:.4f}\n" 
        metrics_text += f" Support: {metrics[c]['support']}\n\n" 

    # Macro/weighted
    macro_precision = np.mean([metrics[c]['precision'] for c in class_names]) 
    macro_recall = np.mean([metrics[c]['recall'] for c in class_names]) 
    macro_f1 = np.mean([metrics[c]['f1'] for c in class_names]) 
    total_support = sum([metrics[c]['support'] for c in class_names]) 
    weighted_precision = sum([metrics[c]['precision'] * metrics[c]['support'] for c in class_names]) / total_support 
    weighted_recall = sum([metrics[c]['recall'] * metrics[c]['support'] for c in class_names]) / total_support 
    weighted_f1 = sum([metrics[c]['f1'] * metrics[c]['support'] for c in class_names]) / total_support 
    
    metrics_text += f"Macro Avg:\n Precision: {macro_precision:.4f}\n Recall: {macro_recall:.4f}\n F1-score: {macro_f1:.4f}\n\n"
    metrics_text += f"Weighted Avg:\n Precision: {weighted_precision:.4f}\n Recall: {weighted_recall:.4f}\n F1-score: {weighted_f1:.4f}" 
    
    axes[2, 1].text(0.05, 0.95, metrics_text, transform=axes[2, 1].transAxes, 
                    fontfamily='monospace', verticalalignment='top', fontsize=9) 
    axes[2, 1].set_title('Performance Metrics') 
    axes[2, 1].set_axis_off()
    
    # 9. Reserved slot
    axes[2, 2].axis("off")
    axes[2, 2].set_title("Reserved")
    
    plt.tight_layout()
    plt.show()
    
    # Print training summary with F1-score focus
    print("=" * 60)
    print("TRAINING SUMMARY (F1-Score Focus)")
    print("=" * 60)
    print(f"Total epochs trained: {len(history.history['loss'])}")
    if 'f1_score' in history.history:
        print(f"Final training F1-score: {history.history['f1_score'][-1]:.4f}")
        print(f"Final validation F1-score: {history.history['val_f1_score'][-1]:.4f}")
        print(f"Best validation F1-score: {max(history.history['val_f1_score']):.4f}")
    print(f"Final training accuracy: {history.history['accuracy'][-1]:.4f}")
    print(f"Final validation accuracy: {history.history['val_accuracy'][-1]:.4f}")
    print(f"Final training loss: {history.history['loss'][-1]:.4f}")
    print(f"Final validation loss: {history.history['val_loss'][-1]:.4f}")
    
    print("\nTest Set Performance:")
    print(classification_report(y_true, y_pred, target_names=class_names))
    
    # Calculate and display overall F1-score
    overall_f1 = f1_score(y_true, y_pred, average='weighted')
    print(f"\nOverall Weighted F1-Score on Test Set: {overall_f1:.4f}")
    
    return fig

def main():
    class_names = ['others', 'sheila']
    n_classes = len(class_names)
    
    # Model with F1-score monitoring
    #model = build_student_model((1600,), n_classes)
    model = create_sparse_friendly_cnn((40, 40, 1), n_classes)
    model.summary()
    
    # Enhanced callbacks with F1-score monitoring
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_f1_score', patience=10, restore_best_weights=True, 
            mode='max', verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_f1_score', factor=0.2, patience=5, min_lr=1e-6, 
            mode='max', verbose=1),
        tf.keras.callbacks.ModelCheckpoint(
            'best_f1_model.h5', monitor='val_f1_score',
            save_best_only=True, mode='max', verbose=1)
    ]

    # Train with class weights
    class_weights = compute_class_weight('balanced', 
                                   classes=np.unique(y_train), 
                                   y=y_train)
    class_weight_dict = dict(enumerate(class_weights))
    
    print("Training with F1-score monitoring...")
    print(f"Class weights: {class_weight_dict}")
    
    history = model.fit(
         X_train, y_train,
         epochs=50, batch_size=32,
         validation_data=(X_val, y_val),
         class_weight=class_weight_dict,
         callbacks=callbacks, verbose=1
    )
    
    plot_training_performance(history, model, X_test, y_test, class_names, y_test_labels=class_names)

    # Save model
    model.save('speech_command_f1_model.h5')
    print("Model saved as 'speech_command_f1_model.h5'")

if __name__ == "__main__":
    main()

2025-09-02 15:27:29.992583: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2025-09-02 15:27:29.992966: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2025-09-02 15:27:29.993708: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2025-09-02 15:27:29.994208: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-09-02 15:27:29.994725: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "teacher_kws_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 40, 40, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 40, 40, 32)     │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 40, 40, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 40, 40, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 20, 20, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 20, 20, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 20, 20, 64)     │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 20, 20, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 20, 20, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 10, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 10, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 10, 10, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 10, 10, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 10, 10, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 5, 5, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 5, 5, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3200)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       819,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_3 (ReLU)                  │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 128)            │           51

 Total params: 955,426 (3.64 MB)

 Trainable params: 954,082 (3.64 MB)

 Non-trainable params: 1,344 (5.25 KB)

Training with F1-score monitoring...
Class weights: {0: 0.571121131379381, 1: 4.015129682997118}
Epoch 1/50


ValueError: Input 0 of layer "teacher_kws_cnn" is incompatible with the layer: expected shape=(None, 40, 40, 1), found shape=(None, 1600)